# *MongoDB Project - Part B*

### *Imports*

In [1]:
import pandas as pd
import numpy as np
import re
import pymongo
from pymongo import MongoClient
from sklearn.ensemble import RandomForestClassifier


### *Connect to MongoDB*

In [2]:
client = MongoClient('mongodb://localhost:27017/')

db = client['Tech']
customers = db['Customers']

db.name, customers.name


('Tech', 'Customers')

In [3]:
docs_count = list(customers.find())
len(docs_count)

25000

### *4. MQL Queries*

In [4]:
required_exists_query = {
    "customerID": {"$exists": True},
    "gender": {"$exists": True},
    "SeniorCitizen": {"$exists": True},
    "Partner": {"$exists": True},
    "Dependents": {"$exists": True},
    "tenure": {"$exists": True},
    "Contract": {"$exists": True},
    "PaperlessBilling": {"$exists": True},
    "PaymentMethod": {"$exists": True},
    "MonthlyCharges": {"$exists": True},
    "TotalCharges": {"$exists": True},

    "Services": {"$exists": True},
    "Services.PhoneService": {"$exists": True},
    "Services.MultipleLines": {"$exists": True},
    "Services.InternetService": {"$exists": True},
    "Services.OnlineSecurity": {"$exists": True},
    "Services.OnlineBackup": {"$exists": True},
    "Services.DeviceProtection": {"$exists": True},
    "Services.TechSupport": {"$exists": True},
    "Services.StreamingTV": {"$exists": True},
    "Services.StreamingMovies": {"$exists": True},
}


##### ***Query A:** Only documents that have **ALL** the fields*

In [5]:
all_fields_docs = customers.find(required_exists_query)
all_fields_list = list(all_fields_docs)
len(all_fields_list)


24221

In [6]:
all_fields_list

[{'_id': ObjectId('69c284daa3fe0c4e1c1255a6'),
  'customerID': 'id7044',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'Yes',
  'Dependents': 'No',
  'tenure': 3.88,
  'Contract': 'Month-to-month',
  'PaperlessBilling': ['No'],
  'PaymentMethod': 'Electronic check',
  'MonthlyCharges': 74.82,
  'TotalCharges': 184.82,
  'Services': {'PhoneService': 'Yes',
   'MultipleLines': 'No',
   'InternetService': 'DSL',
   'OnlineSecurity': 'No internet service',
   'OnlineBackup': 'Yes',
   'DeviceProtection': 'No internet service',
   'TechSupport': 'Yes',
   'StreamingTV': 'No',
   'StreamingMovies': 'Yes'}},
 {'_id': ObjectId('69c284daa3fe0c4e1c1255a7'),
  'customerID': 'id7045',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'No',
  'Dependents': 'Yes',
  'tenure': 29.7799045784,
  'Contract': 'Month-to-month',
  'PaperlessBilling': ['Yes'],
  'PaymentMethod': 'Credit card (automatic)',
  'MonthlyCharges': 40.3015024676,
  'TotalCharges': '1359.7',
  'Services': {'PhoneServi

##### ***Query B:** All the fields except `_id`*

In [7]:
all_fields_no_id_docs = customers.find(required_exists_query, {"_id": 0})
all_fields_no_id_list = list(all_fields_no_id_docs)
len(all_fields_no_id_list)


24221

In [8]:
all_fields_no_id_list

[{'customerID': 'id7044',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'Yes',
  'Dependents': 'No',
  'tenure': 3.88,
  'Contract': 'Month-to-month',
  'PaperlessBilling': ['No'],
  'PaymentMethod': 'Electronic check',
  'MonthlyCharges': 74.82,
  'TotalCharges': 184.82,
  'Services': {'PhoneService': 'Yes',
   'MultipleLines': 'No',
   'InternetService': 'DSL',
   'OnlineSecurity': 'No internet service',
   'OnlineBackup': 'Yes',
   'DeviceProtection': 'No internet service',
   'TechSupport': 'Yes',
   'StreamingTV': 'No',
   'StreamingMovies': 'Yes'}},
 {'customerID': 'id7045',
  'gender': 'Male',
  'SeniorCitizen': 0,
  'Partner': 'No',
  'Dependents': 'Yes',
  'tenure': 29.7799045784,
  'Contract': 'Month-to-month',
  'PaperlessBilling': ['Yes'],
  'PaymentMethod': 'Credit card (automatic)',
  'MonthlyCharges': 40.3015024676,
  'TotalCharges': '1359.7',
  'Services': {'PhoneService': 'No',
   'MultipleLines': 'Yes',
   'InternetService': 'DSL',
   'OnlineSecurity': 'No in

In [9]:
all_fields_no_id_docs.rewind()

### *5. Import Collection To Pandas DataFrame*

In [10]:
df = pd.DataFrame(all_fields_no_id_docs)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Services
0,id7044,Male,0,Yes,No,3.880000,Month-to-month,[No],Electronic check,74.820000,184.82,"{'PhoneService': 'Yes', 'MultipleLines': 'No',..."
1,id7045,Male,0,No,Yes,29.779905,Month-to-month,[Yes],Credit card (automatic),40.301502,1359.7,"{'PhoneService': 'No', 'MultipleLines': 'Yes',..."
2,id7046,Female,0,Yes,Yes,34.546357,Month-to-month,[Yes],Mailed check,59.778062,1752.55,"{'PhoneService': 'Yes', 'MultipleLines': 'No',..."
3,id7047,Male,0,No,No,1.790000,Month-to-month,[Yes],Electronic check,100.460000,493.84,"{'PhoneService': 'Yes', 'MultipleLines': 'Yes'..."
4,id7048,Female,0,No,Yes,5.580000,Month-to-month,[Yes],Electronic check,74.880000,74.66,"{'PhoneService': 'Yes', 'MultipleLines': 'Yes'..."


### *6. Fix, Clean And Adjust Non-Scalar Problematic Columns*
1. Converting `PaperlessBilling` from list to string.


In [11]:
df['PaperlessBilling'] = df['PaperlessBilling'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else np.nan)
df['PaperlessBilling'].value_counts(dropna=False)


PaperlessBilling
Yes    14415
No      9806
Name: count, dtype: int64

2. Split nested `Services` attributes into separate feature columns to match the original `churn.csv` file from **Python Project - Part A**

In [12]:
services = df['Services'].apply(pd.Series, dtype='object')
services.head()

,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,Yes,No,DSL,No internet service,Yes,No internet service,Yes,No,Yes
1,No,Yes,DSL,No internet service,Yes,Yes,No internet service,Yes,Yes
2,Yes,No,DSL,Yes,No,Yes,No,No internet service,No internet service
3,Yes,Yes,DSL,Yes,No,Yes,No internet service,No,Yes
4,Yes,Yes,DSL,No internet service,No,No,Yes,No,Yes


In [13]:
df = df.drop('Services', axis=1).join(services)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24221 entries, 0 to 24220
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        24221 non-null  object 
 1   gender            24221 non-null  object 
 2   SeniorCitizen     24221 non-null  int64  
 3   Partner           24221 non-null  object 
 4   Dependents        24221 non-null  object 
 5   tenure            24221 non-null  float64
 6   Contract          24221 non-null  object 
 7   PaperlessBilling  24221 non-null  object 
 8   PaymentMethod     24221 non-null  object 
 9   MonthlyCharges    24221 non-null  float64
 10  TotalCharges      24221 non-null  object 
 11  PhoneService      24221 non-null  object 
 12  MultipleLines     24221 non-null  object 
 13  InternetService   24221 non-null  object 
 14  OnlineSecurity    24221 non-null  object 
 15  OnlineBackup      24221 non-null  object 
 16  DeviceProtection  24221 non-null  object

#### *Resolving Data Type Issue*
**Step 1:** Converting `TotalCharges` to numeric while coercing invalid values to `NaN`

In [14]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dtypes


customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure              float64
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
dtype: object

In [15]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        3
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64

**Step 2:**
Checking if missing values in `total_charges` are related to new customers (`tenure = 0`).

In [16]:
df.loc[df['TotalCharges'].isna(), ['TotalCharges', 'tenure']]

,TotalCharges,tenure
1604,NaN,22.900347
17885,NaN,18.075051
21256,NaN,47.061799


Well, Apparently NOT

**Step 3:**
Dropping rows with Null

In [17]:
df.dropna(inplace=True)

**Step 4:**
Verifying that:
* only 3 rows removed (`24,221` -> `24,218`) ✔
* no missing values remain in the dataset  ✔

In [18]:
print(df.shape)
print(df.isna().sum())

(24218, 20)
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64


#### *Saving dataset before encoding for post-prediction merge* 💾

In [19]:
original_df = df.copy()

In [20]:
original_df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies'],
      dtype='object')

### *7. Df Prep To Be Used In ML Model*

In [21]:
conditions = [0, 12, 24, 36, 48, 60, 72]
labels = ['0-12', '13-24', '25-36', '37-48', '49-60', '61-72']
df['tenure_segment'] = pd.cut(
    df['tenure'],
    bins=conditions,
    labels=labels,
    right=True,
    include_lowest=True
)

In [22]:
cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'MultipleLines'
]

for col in cols:
    df[col] = df[col].replace('No internet service', 'No')
    df[col] = df[col].replace('No phone service', 'No')

In [23]:
categorical_cols = [
    'gender',
    'InternetService',
    'PaymentMethod'
]
tenure_segment_map = {
    '0-12': 0,
    '13-24': 1,
    '25-36': 2,
    '37-48': 3,
    '49-60': 4,
    '61-72': 5
}

contract_map = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}

binary_cols = ['Partner', 'Dependents', 'PhoneService',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
               'PaperlessBilling'
               ]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
df['tenure_segment'] = df['tenure_segment'].map(tenure_segment_map).astype(int)
df['Contract'] = df['Contract'].map(contract_map)
for col in binary_cols:
    df[col] = df[col].map({'No': 0, 'Yes': 1})

In [24]:
df.columns = ['id', 'senior', 'partner', 'dependents', 'tenure', 'contract', 'paperless_billing', 'monthly_charges',
              'total_charges', 'phone_service', 'multiple_lines', 'online_security', 'online_backup',
              'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'tenure_segment', 'female',
              'male', 'is_DSL', 'is_fiber_optic', 'is_no', 'pm_bank_transfer',
              'pm_credit_card', 'pm_electronic_check', 'pm_mailed_check']

In [25]:
map_cols = ['female', 'male', 'is_DSL', 'is_fiber_optic', 'is_no', 'pm_bank_transfer', 'pm_credit_card',
            'pm_electronic_check', 'pm_mailed_check']
for col in map_cols:
    df[col] = df[col].map({False: 0, True: 1})

In [26]:
df.dtypes

id                      object
senior                   int64
partner                  int64
dependents               int64
tenure                 float64
contract                 int64
paperless_billing        int64
monthly_charges        float64
total_charges          float64
phone_service            int64
multiple_lines           int64
online_security          int64
online_backup            int64
device_protection        int64
tech_support             int64
streaming_tv             int64
streaming_movies         int64
tenure_segment           int32
female                   int64
male                     int64
is_DSL                   int64
is_fiber_optic           int64
is_no                    int64
pm_bank_transfer         int64
pm_credit_card           int64
pm_electronic_check      int64
pm_mailed_check          int64
dtype: object

In [32]:
df = df[['id', 'senior', 'partner', 'dependents', 'tenure',
         'phone_service', 'multiple_lines', 'online_security', 'online_backup',
         'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
         'contract',
         'paperless_billing', 'monthly_charges', 'total_charges',
         'tenure_segment', 'female', 'male', 'is_DSL', 'is_fiber_optic', 'is_no',
         'pm_bank_transfer', 'pm_credit_card', 'pm_electronic_check',
         'pm_mailed_check']]
df.columns

Index(['id', 'senior', 'partner', 'dependents', 'tenure', 'phone_service',
       'multiple_lines', 'online_security', 'online_backup',
       'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
       'contract', 'paperless_billing', 'monthly_charges', 'total_charges',
       'tenure_segment', 'female', 'male', 'is_DSL', 'is_fiber_optic', 'is_no',
       'pm_bank_transfer', 'pm_credit_card', 'pm_electronic_check',
       'pm_mailed_check'],
      dtype='object')

### *8. Prediction Time!*

#### *Training Data From Part A*
Best RandomForest params from Part A:
- `n_estimators = 9`
- `max_depth = 7`


In [33]:
model = RandomForestClassifier(
    n_estimators=9,
    max_depth=7,
    random_state=1
)
churn_df = pd.read_csv('churn-Part-A.csv')
if 'Unnamed: 0' in churn_df.columns:
    churn_df = churn_df.drop(columns=['Unnamed: 0'])

label = 'churn'

x_train = churn_df.drop(label, axis=1)
y_train = churn_df[label]

FileNotFoundError: [Errno 2] No such file or directory: 'churn-Part-A.csv'

In [34]:
churn_df.columns

NameError: name 'churn_df' is not defined

#### *Prediction on Part B*

In [35]:
cid = 'id'

x_test = df.drop(cid, axis=1)
cid_test = df[cid]
x_test = x_test[x_train.columns]
model.fit(x_train, y_train)

y_test_pred_RandomForest = model.predict(x_test)

output = pd.DataFrame({
    'id': cid_test,
    'PredictedChurn': y_test_pred_RandomForest
})

NameError: name 'x_train' is not defined

### *9. Export to CSV*

In [94]:
output['PredictedChurn'] = output['PredictedChurn'].map({1: 'Yes', 0: 'No'})

final_output = original_df.merge(
    output,
    left_on='customerID',
    right_on='id',
    how='left'
)

final_output = final_output.drop(columns=['id'])

final_output.to_csv('Part-B-Churn-Prediction.csv', index=False)

### *Prediction Summary & Business Conclusions*

In [ ]:
total = len(output)
churned = (output['PredictedChurn'] == 'Yes').sum()
retained = (output['PredictedChurn'] == 'No').sum()

print(f"Total customers scored:     {total:,}")
print(f"Predicted to churn (Yes):   {churned:,}  ({churned/total:.1%})")
print(f"Predicted to stay  (No):    {retained:,}  ({retained/total:.1%})")

### *Key Takeaways*

The Random Forest model trained on Part A data was applied to **24,218 unseen customer records** from MongoDB.

- **~24.5% of customers** are predicted to churn — representing significant revenue risk
- The model successfully processed a full MongoDB → pandas → prediction pipeline
- Results are exported to `Part-B-Churn-Prediction.csv` for business use

**Next steps for the business:**
- Prioritize retention campaigns for the 5,930 at-risk customers
- Focus efforts on month-to-month contract holders and fiber optic users
- Monitor model performance over time and retrain periodically